In [1]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, accuracy_score, precision_recall_curve, average_precision_score,
    recall_score, make_scorer
)
from sklearn.preprocessing import label_binarize
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime
from pathlib import Path
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate


In [2]:
# === CONFIGURACION ===
REPO_ROOT = Path(".").resolve()
RUN_ID = datetime.today().strftime("%Y%m%d")

INPUT_BASE = REPO_ROOT / "data" / "intermediate" / "05_seleccion"
OUTPUT_BASE = REPO_ROOT / "reports" / "09_modelo_lightgbm" / RUN_ID

MODEL_NAME = "LightGBM"
INTENTO = 10  # Cambia segun version
N_SPLITS = 3
TARGET_CLASS = None  # usa el minimo de nivel_triage si es None
N_ESTIMATORS = 180
LEARNING_RATE = 0.05
fecha_actual = datetime.today().strftime("%Y-%m-%d")

if not INPUT_BASE.exists():
    raise FileNotFoundError("No se encontro data/intermediate/05_seleccion")

input_candidates = sorted([p for p in INPUT_BASE.iterdir() if p.is_dir()])
if not input_candidates:
    raise FileNotFoundError("No se encontraron subcarpetas en data/intermediate/05_seleccion")
INPUT_PATH = input_candidates[-1]
print(f"Usando input: {INPUT_PATH}")

OUTPUT_PATH = OUTPUT_BASE / f"{MODEL_NAME}_{INTENTO:02d}_{fecha_actual}"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

BALANCE_METHODS = {
    "NONE": None,
    "SMOTE": SMOTE(random_state=42),
    "UNDER": RandomUnderSampler(random_state=42),
    "SMOTEENN": SMOTEENN(random_state=42)
}

metricas_totales = []

# Detectar variantes de datasets
variantes_X = sorted([f for f in os.listdir(INPUT_PATH) if f.startswith("X_train")])


Usando input: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\05_seleccion\04_2026_01_12


In [3]:
best_score = -1.0
best_info = None


def build_lgbm(num_classes: int):
    "Modelo liviano para multi/binary clase."
    params = dict(
        boosting_type="gbdt",
        objective="multiclass" if num_classes > 2 else "binary",
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        num_leaves=31,
        max_depth=-1,
        subsample=0.85,
        colsample_bytree=0.9,
        max_bin=63,
        min_child_samples=20,
        reg_alpha=0.1,
        reg_lambda=1.0,
        bagging_freq=1,
        n_jobs=-1,
        random_state=42,
    )
    if num_classes > 2:
        params["num_class"] = num_classes
    return LGBMClassifier(**params)


def build_pipeline(balancer, num_classes: int):
    steps = []
    if balancer is not None:
        steps.append(("balance", balancer))
    steps.append(("model", build_lgbm(num_classes)))
    return Pipeline(steps)


def resolve_target_class(y, target):
    classes = list(pd.Series(y).unique())
    if target in classes:
        return target
    if str(target) in [str(c) for c in classes]:
        for c in classes:
            if str(c) == str(target):
                return c
    print(f"WARN: TARGET_CLASS {target} no esta en clases; se usa {classes[0]}")
    return classes[0]


for balance_name, balancer in BALANCE_METHODS.items():
    print(f"=== Balanceo: {balance_name} ===")
    output_subdir = OUTPUT_PATH / balance_name
    output_subdir.mkdir(parents=True, exist_ok=True)

    for x_file in variantes_X:
        base_name = x_file.replace("X_train_", "").replace(".parquet", "")
        try:
            print(f"Procesando: {base_name}")

            # Cargar datos
            X_train = pd.read_parquet(INPUT_PATH / f"X_train_{base_name}.parquet")
            X_test = pd.read_parquet(INPUT_PATH / f"X_test_{base_name}.parquet")
            y_train = pd.read_parquet(INPUT_PATH / f"y_train_{base_name}.parquet").squeeze()
            y_test = pd.read_parquet(INPUT_PATH / f"y_test_{base_name}.parquet").squeeze()

            nan_total_train = int(X_train.isna().sum().sum())
            nan_total_test = int(X_test.isna().sum().sum())
            if nan_total_train > 0 or nan_total_test > 0:
                print(f"  NaN - train: {nan_total_train}, test: {nan_total_test}")

            if pd.Series(y_train).nunique() < 2:
                print("  Dataset con una sola clase en train; se omite.")
                continue

            target_val = TARGET_CLASS if TARGET_CLASS is not None else pd.Series(y_train).min()
            target_class = resolve_target_class(y_train, target_val)

            # Ajustar etiquetas para LightGBM (deben empezar en 0)
            y_min = y_train.min()
            y_train_adj = y_train - y_min
            y_test_adj = y_test - y_min
            target_class_adj = target_class - y_min
            num_classes = int(pd.Series(y_train_adj).nunique())

            # CV en train (sin tocar test)
            cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
            model_cv = build_pipeline(balancer, num_classes)

            target_scorer = make_scorer(
                recall_score,
                labels=[target_class_adj],
                average="macro",
                zero_division=0
            )
            scorers = {
                "recall_target": target_scorer,
                "f1_macro": "f1_macro"
            }
            cv_results = cross_validate(
                model_cv,
                X_train,
                y_train_adj,
                cv=cv,
                scoring=scorers
            )

            cv_recall_scores = cv_results["test_recall_target"]
            cv_macro_f1_scores = cv_results["test_f1_macro"]
            cv_recall_target = float(np.mean(cv_recall_scores))
            cv_macro_f1 = float(np.mean(cv_macro_f1_scores))

            df_cv = pd.DataFrame({
                "fold": range(1, len(cv_recall_scores) + 1),
                "cv_recall_target": cv_recall_scores,
                "cv_macro_f1": cv_macro_f1_scores,
                "nan_total_train": [nan_total_train] * len(cv_recall_scores),
                "nan_total_test": [nan_total_test] * len(cv_recall_scores)
            })
            df_cv.to_csv(output_subdir / f"cv_metricas_{base_name}.csv", index=False)

            resumen_cv = {
                "modelo": base_name,
                "balanceo": balance_name,
                "cv_recall_target": cv_recall_target,
                "cv_macro_f1": cv_macro_f1,
                "nan_total_train": nan_total_train,
                "nan_total_test": nan_total_test
            }
            metricas_totales.append(resumen_cv)

            if (cv_recall_target > best_score) or (cv_recall_target == best_score and cv_macro_f1 > (best_info.get("cv_macro_f1", -1.0) if best_info else -1.0)):
                best_score = cv_recall_target
                best_info = {
                    "balanceo": balance_name,
                    "base_name": base_name,
                    "cv_recall_target": cv_recall_target,
                    "cv_macro_f1": cv_macro_f1
                }

            print(f"  CV recall clase {target_class}: {cv_recall_target:.3f} | CV macro F1: {cv_macro_f1:.3f}")

        except Exception as e:
            print(f" Error en {base_name} con balanceo {balance_name}: {e}")


if best_info is None:
    raise RuntimeError("No se pudo seleccionar un modelo ganador; revisa los logs.")

print(f"\nMejor modelo por CV (recall clase {TARGET_CLASS}): {best_info}")

# Entrenar y evaluar SOLO el ganador en test
balance_name = best_info["balanceo"]
base_name = best_info["base_name"]
balancer = BALANCE_METHODS[balance_name]

output_subdir = OUTPUT_PATH / balance_name
output_subdir.mkdir(parents=True, exist_ok=True)

X_train = pd.read_parquet(INPUT_PATH / f"X_train_{base_name}.parquet")
X_test = pd.read_parquet(INPUT_PATH / f"X_test_{base_name}.parquet")
y_train = pd.read_parquet(INPUT_PATH / f"y_train_{base_name}.parquet").squeeze()
y_test = pd.read_parquet(INPUT_PATH / f"y_test_{base_name}.parquet").squeeze()

y_min = y_train.min()
y_train_adj = y_train - y_min
y_test_adj = y_test - y_min
num_classes = int(pd.Series(y_train_adj).nunique())

start = time.time()
model = build_pipeline(balancer, num_classes)
model.fit(X_train, y_train_adj)
tiempo = (time.time() - start) / 60

y_pred_train_adj = model.predict(X_train)
y_pred_test_adj = model.predict(X_test)
y_proba = model.predict_proba(X_test)

y_pred_train = y_pred_train_adj + y_min
y_pred_test = y_pred_test_adj + y_min

class_order_adj = model.named_steps["model"].classes_
class_order = class_order_adj + y_min

report_test = pd.DataFrame(classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)).T
report_train = pd.DataFrame(classification_report(y_train, y_pred_train, output_dict=True, zero_division=0)).T
report_test["set"] = "test"
report_train["set"] = "train"
full_report = pd.concat([report_train, report_test])
full_report["tiempo_min"] = tiempo
full_report.to_csv(output_subdir / f"metricas_{base_name}_BEST.csv")

cm = confusion_matrix(y_test, y_pred_test, labels=class_order)
with PdfPages(output_subdir / f"reporte_{base_name}_BEST.pdf") as pdf:
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_order).plot(cmap="Blues")
    plt.title("Matriz de Confusion")
    pdf.savefig(); plt.close()

    if np.unique(y_test).size >= 2:
        y_bin = label_binarize(y_test_adj, classes=class_order_adj)
        if y_bin.shape[1] == 1:
            y_bin = np.hstack([1 - y_bin, y_bin])
        proba_for_curves = y_proba
        if proba_for_curves.shape[1] == 1:
            proba_for_curves = np.hstack([1 - y_proba, y_proba])

        plt.figure()
        for i, clase in enumerate(class_order):
            fpr, tpr, _ = roc_curve(y_bin[:, i], proba_for_curves[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f"Clase {clase} (AUC={roc_auc:.2f})")
        plt.plot([0, 1], [0, 1], "k--")
        plt.title("Curvas ROC por clase")
        plt.xlabel("FPR")
        plt.ylabel("TPR")
        plt.legend()
        pdf.savefig(); plt.close()

        plt.figure()
        for i, clase in enumerate(class_order):
            precision, recall, _ = precision_recall_curve(y_bin[:, i], proba_for_curves[:, i])
            pr_auc = average_precision_score(y_bin[:, i], proba_for_curves[:, i])
            plt.plot(recall, precision, label=f"Clase {clase} (PR AUC={pr_auc:.2f})")
        plt.title("Curvas Precision-Recall por clase")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.legend()
        pdf.savefig(); plt.close()
    else:
        print("  Test con 1 clase; se omiten curvas ROC/PR.")

resumen = {
    "modelo": base_name,
    "balanceo": balance_name,
    "accuracy_test": accuracy_score(y_test, y_pred_test),
    "macro_f1_test": report_test.loc["macro avg", "f1-score"],
    "weighted_f1_test": report_test.loc["weighted avg", "f1-score"],
    "accuracy_train": accuracy_score(y_train, y_pred_train),
    "macro_f1_train": report_train.loc["macro avg", "f1-score"],
    "weighted_f1_train": report_train.loc["weighted avg", "f1-score"],
    "tiempo_min": tiempo,
    "sobreajuste_macro_f1": report_train.loc["macro avg", "f1-score"] - report_test.loc["macro avg", "f1-score"],
    "cv_recall_target": best_info["cv_recall_target"],
    "cv_macro_f1": best_info["cv_macro_f1"]
}
for clase in report_test.index:
    if clase not in ["accuracy", "macro avg", "weighted avg"]:
        resumen[f"f1_{clase}_test"] = report_test.loc[clase, "f1-score"]
        resumen[f"recall_{clase}_test"] = report_test.loc[clase, "recall"]
        resumen[f"precision_{clase}_test"] = report_test.loc[clase, "precision"]
        resumen[f"f1_{clase}_train"] = report_train.loc[clase, "f1-score"]
        resumen[f"recall_{clase}_train"] = report_train.loc[clase, "recall"]
        resumen[f"precision_{clase}_train"] = report_train.loc[clase, "precision"]

metricas_totales.append(resumen)
print(f"\nBEST {base_name} [{balance_name}]: recall_target_cv={best_info['cv_recall_target']:.3f}, F1_macro_test={resumen.get('macro_f1_test', 0):.3f}")

# Consolidar todas las metricas en un CSV unico
if metricas_totales:
    df_metricas = pd.DataFrame(metricas_totales)
    df_metricas.to_csv(OUTPUT_PATH / "resumen_modelos_lightgbm.csv", index=False)
    print(f"Se guardo resumen_modelos_lightgbm.csv con {len(df_metricas)} filas en {OUTPUT_PATH}")
    display(df_metricas.head())
else:
    print("No se genero ninguna metrica; revisa los logs de arriba.")


=== Balanceo: NONE ===
Procesando: MaxAbs_FE_ALL
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.655027 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14713
[LightGBM] [Info] Number of data points in the train set: 298925, number of used features: 1146
[LightGBM] [Info] Start training from score -2.379916
[LightGBM] [Info] Start training from score -1.639819
[LightGBM] [Info] Start training from score -1.547894
[LightGBM] [Info] Start training from score -1.498879
[LightGBM] [Info] Start training from score -1.282473
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 wil

,modelo,balanceo,cv_recall_target,cv_macro_f1,nan_total_train,nan_total_test,accuracy_test,macro_f1_test,weighted_f1_test,accuracy_train,...,precision_4_test,f1_4_train,recall_4_train,precision_4_train,f1_5_test,recall_5_test,precision_5_test,f1_5_train,recall_5_train,precision_5_train
0,MaxAbs_FE_ALL,NONE,0.056286,0.404995,0.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MaxAbs_FE_ANOVA,NONE,0.000771,0.241351,0.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MaxAbs_FE_MI,NONE,0.000795,0.275272,0.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MaxAbs_FE_PCA30,NONE,0.012698,0.349338,0.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MaxAbs_FE_PCAopt,NONE,0.052720,0.402608,0.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
